# tobac workflow on ICON - final application full period 
- **used for application to full 30 years with runner/starter 29**
- duplicate of 25_interactive but adapted for runner (without plotting etc.)
- what this nbook does:
    - Load ICON ngc4008, resampled to IMERG grid, tropical belt selection
    - tobac application:
        - anvils 
            - detect on Tb (Nasa-Tb)
            - segment on Tb 
            - bulk stats of IWP (CCIC)
            - bulk stats on Tb
            - additional stats (area)
        - precip cells
            - detect on precip
            - segment on precip
            - bulk stats of Precip (precip)
            - additional stats (area)
        - merge anvil properties to cell features table to df_cell_combined
        - add additional columns
        - save **df_cell_combined**
- input:
    - ICON pr, BT, IWP-related variables
    - date YYYY-MM-DD-HHMM
- adapt:
    - output dir for seg results data
    - tobac parameters for detection and segmentation
    - date
- output: 
    - combined precip features df with anvil properties and further columns (FINAL PRODUCT)


In [38]:
from pathlib import Path
import sys
# set date
if "launcher" in sys.argv[0]: 
    date ="2031-06-23-0600" #"2048-09-29-0600"# "2023-06-06-2100"
    interactive =True
else: 
    date = sys.argv[1]
    interactive = False
print(date)

outfiletest = Path(f"../output/apply_seg/apply_ICON/FULL/cell_features_combined/cell_features_combined_anvil_detectthres225_seg241_cell_detectthres10_seg10_{date}.h5")

filedoesnotexist = not outfiletest.exists() 
filedoesnotexist

2031-06-23-0600


In [2]:
if filedoesnotexist: 
    import warnings
    
    # ignore general categories
    for cat in [FutureWarning, RuntimeWarning, UserWarning]:
        warnings.filterwarnings("ignore", category=cat)
    
    # ignore specific repeated messages
    for msg in [
        "pyproj unable to set PROJ database path",
        "As of v1.6.0, segmentation with time length 1",
        "Using DEFAULT_SPHERICAL_EARTH_RADIUS"
    ]:
        warnings.filterwarnings("ignore", message=msg)

In [3]:
if filedoesnotexist:
    import numpy as np
    import xarray as xr
    import os
    
    from pathlib import Path
    
    # Import tobac itself:
    import tobac
    import tobac.utils
    
    print("using tobac version", str(tobac.__version__))
    
    import iris
    
    # plotting
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import seaborn as sns
    
    # load tools
    import sys 
    sys.path.append("../tools")
    
    
    import handle_ICON_data as ic
    
    from defs_application import georegions 
    import use_tobac
    import plots_mapping
    
    
    import dask

using tobac version 1.6.3


# arguments

In [4]:
if filedoesnotexist:    
    
    # threshold_anvils
    detection_threshold_anvil = 225 # K
    multi_detection_thresholds_anvil =  [225]  
    segmentation_threshold_anvil = 241 
    
    # threhold precip
    detection_threshold_cell = 10
    multi_detection_thresholds_cell = [10] 
    segmentation_threshold_cell = 10
    
    # build tag from parameters
    param_tag = f"anvil_detectthres{detection_threshold_anvil}_seg{segmentation_threshold_anvil}_cell_detectthres{detection_threshold_cell}_seg{segmentation_threshold_cell}"
    #param_tag

In [5]:
if filedoesnotexist:
    ############# ANVILS ############
    # detection para (Tb)
    feature_detection_params_anvil = dict()
    feature_detection_params_anvil["threshold"] = multi_detection_thresholds_anvil
    feature_detection_params_anvil["target"] = "minimum"
    feature_detection_params_anvil["position_threshold"] = "weighted_diff"
    feature_detection_params_anvil["PBC_flag"]='hdim_2'
    
    
    
    # Gaussian smoothing before detection
    feature_detection_params_anvil["sigma_threshold"] = 0.5 # default
    
    # from below 
    dX = 11119.492664455873
    dxy =  dX 
    
    # minimum number of pixels below threshold_anvil
    #feature_detection_params_anvil["n_min_threshold"] = 3 # one pixel approx 10km * 10km = 100km2
    
    # minimum pixel distance to next feature
    feature_detection_params_anvil["min_distance"]= dxy*20  # mcs ICON trakcing example *4 => aber wir haben geresamplet zu 11 x 11km 

In [6]:
if filedoesnotexist: 
    ############# CELLS ############
    # detection para (Tb)
    feature_detection_params_cell = dict()
    feature_detection_params_cell["threshold"] = multi_detection_thresholds_cell
    feature_detection_params_cell["target"] = "maximum"
    feature_detection_params_cell["position_threshold"] = "weighted_diff"
    feature_detection_params_cell["PBC_flag"]='hdim_2'
    
    
    # Gaussian smoothing before detection
    feature_detection_params_cell["sigma_threshold"] = 0.05
    #feature_detection_params_cell["n_min_threshold"] = 3 # brueck


In [7]:
if filedoesnotexist: 
    savedir = Path("../output/apply_seg/apply_ICON/FULL")  # tobac_tuning_versions_ICON
    
    
    # create main directory
    savedir.mkdir(parents=True, exist_ok=True)
    
    # define subdirectories
    subdirs = [
        "cell_features_combined",
    ]
    
    # create all subdirectories
    for sub in subdirs:
        (savedir / sub).mkdir(parents=True, exist_ok=True)


# load data

In [8]:
if filedoesnotexist: 
    # data selection
    zoom = 9 # approx. 14km, => IMERG is 11km
    vlist = ["pr", "rlut",  "qsvi","clivi","qgvi","crs"] #"tas", # tas
    
    # load data
    datacata = ic.open_ngc4008(zoom = zoom) 
    
    # select timestamp and variables
    dataset = ic.select_time(datacata, date)
    dataset = ic.select_vars(dataset, vlist)
    
    # calculate input variables 
    dataset = ic.add_input_vars(dataset)
    dataset
    
    
    # calculate idx before, once so that and get_nn_lon_lat_index and imerg ref image is not loaded 4 times
    imerg_grid = xr.open_dataset("../output/aux/imerg_grid.nc")
    lons = imerg_grid.longitude.values
    lats = imerg_grid.latitude.values
    
    idx = ic.get_nn_lon_lat_index(2**zoom, lons, lats) 
    
    # # resample to Imerg grid 
    # Load all variables in one compute call
    tb, precip, iwp = dask.compute( # tas, # tas
       # ic.resample_to_lon_lat(dataset, "tas",           zoom=zoom, to_imerg=False, idx=idx), # tas
        ic.resample_to_lon_lat(dataset, "tb_11",         zoom=zoom, to_imerg=False, idx=idx),
        ic.resample_to_lon_lat(dataset, "rain_tot_rate", zoom=zoom, to_imerg=False, idx=idx),
        ic.resample_to_lon_lat(dataset, "iwp",           zoom=zoom, to_imerg=False, idx=idx),
    )
    
    #print(dataset["tas"].chunks)

# equivalent grid size in meters

In [9]:
# IMERG resolution

# dlon = 0.1   # degrees
# dlat = 0.1   # degrees

# lat0 = precip.latitude.mean().item()   

# R_earth = 6371e3

# dlambda = np.deg2rad(dlon)
# dphi     = np.deg2rad(dlat)
# phi0     = np.deg2rad(lat0)

# dA = R_earth**2 * np.cos(phi0) * dlambda * dphi
# dX = np.sqrt(dA)

# dX

# dX = 11119.492664455873
# dxy =  dX 

# Prepare Data
In the next steps, we
- (regular grid)
- resmaple to imerg data so everything has same resolution (for bulk statiustics needed) 
- convert the dataset into an Iris Cube (which is taken as tobac input)


In [10]:
if filedoesnotexist: 
    # to iris cubes
    tb_iris =tb.to_iris()
    precip_iris = precip.to_iris()
    iwp_iris = iwp.to_iris()
    
    #tas_iris = tas.to_iris() # tas 
    
    # # expand time dim for tobac input (collapsed in the loading functions)
    # import iris
    tb_iris = iris.util.new_axis(tb_iris, scalar_coord='time')
    precip_iris = iris.util.new_axis(precip_iris, scalar_coord='time')
    iwp_iris = iris.util.new_axis(iwp_iris, scalar_coord='time')

# add iwp layer

In [11]:
if filedoesnotexist: 
    # Parse the string into a datetime object
    from datetime import datetime, timedelta
    
    dt = datetime.strptime(date, '%Y-%m-%d-%H%M')
    dt_minus_3h = dt - timedelta(hours=3)
    date_previous = dt_minus_3h.strftime('%Y-%m-%d-%H%M')
    
    # load data
    # data selection
    vlist_tend= ["qsvi","clivi","qgvi","crs"]
    
    
    # select timestamp and variables
    dataset = ic.select_time(datacata, date_previous)
    dataset = ic.select_vars(dataset, vlist_tend)
    
    # calculate input variables 
    dataset = ic.add_iwp(dataset)
    dataset
    
    # Load all variables in one compute call
    iwp = ic.resample_to_lon_lat(dataset, "iwp",           zoom=zoom, to_imerg=False, idx=idx)
    
    #print(
    
    # prepare data
    # to iris cubes
    iwp_iris_previous = iwp.to_iris()
    
    # # expand time dim for tobac input (collapsed in the loading functions)
    iwp_iris_previous = iris.util.new_axis(iwp_iris_previous, scalar_coord='time')
    
    
    
    #####################
    ##### iwp tendency calc
    # Remove time coord from both before arithmetic (or equalize them)
    iwp_iris_previous_notime = iwp_iris_previous.copy()
    iwp_iris_previous_notime.remove_coord('time')
    
    iwp_iris_notime = iwp_iris.copy()
    iwp_iris_notime.remove_coord('time')
    
    # actual calc
    dtiwp = (iwp_iris_notime - iwp_iris_previous_notime) / 3  # d iwp = kg/m2 pro 3h => durch 3 um stündliche Veränderung zu bekommen
    # positive
    dtiwp.data = np.where(dtiwp.data > 0, dtiwp.data, 0)
    
    # Reattach the *current* timestamp's time coord to the result
    # after subtraction, dtiwp already has the right shape (time_dim, lat, lon)
    dtiwp.add_dim_coord(iwp_iris.coord('time'), 0)  # bind as DIM coord on dim 0, not a new axis
    
    
    # rename so bulk stats putput is nice 
    dtiwp.rename("dtiwp")          # sets long_name -> shows as "dtiwp" instead of "Unknown"
    dtiwp.units = "kg m-2 h-1"     # correct units after the /3 (per-hour rate)
    dtiwp.var_name = "dtiwp"
    
    #dtiwp
    
    # # as a side thing bc of added iwp values on cell mask
    iwp_iris.rename("iwp")          # sets long_name -> shows as "dtiwp" instead of "Unknown"
    iwp_iris.var_name = "iwp"
    
    # dtiwp

# Anvil detection and segmentation
## 1.1 tobac detection on Tb image
- use multiple threshold on BT data
- from here on: same workflow for ICON and OBS

### Perform feature detection

In [13]:
if filedoesnotexist: 
    # Perform feature detection:
    print('starting anvil feature detection')
    features = tobac.feature_detection.feature_detection_multithreshold(
        tb_iris, dxy, **feature_detection_params_anvil
    )
    
    print('anvil feature detection performed')

starting anvil feature detection
anvil feature detection performed


## 1.2. Segmentation of Anvils (on BT)
- first filter!

### Filter results (remainer)

In [14]:
if filedoesnotexist: 
    # filter for thres
    filtered_features = features[features["threshold_value"] <= detection_threshold_anvil]


### Set segmentation parameters

In [15]:
if filedoesnotexist: 
    segmentation_params_anvil = dict()
    segmentation_params_anvil["method"] = "watershed"
    segmentation_params_anvil["threshold"] = segmentation_threshold_anvil
    segmentation_params_anvil["target"] = "minimum"
    segmentation_params_anvil["PBC_flag"]='hdim_2'


### perform segmentation

In [16]:
if filedoesnotexist: 
    # Perform segmentation:
    print('Starting anvil segmentation.')
    segmentation_mask_anvil, segmentation_features_anvil = tobac.segmentation.segmentation(
        filtered_features, tb_iris, dxy=dxy, **segmentation_params_anvil
    )
    print('segmentation anvil performed')

Starting anvil segmentation.
segmentation anvil performed


## 1.3. bulk statistics as post processing

In [17]:
if filedoesnotexist: 
    # calc bulk plus area
    df_merged_anvil = use_tobac.add_bulk_stats_anvil(segmentation_features_anvil, segmentation_mask_anvil, tb_iris,precip_iris,iwp_iris)
    
    
    
    # d eq
    df_merged_anvil["d_eq"] = use_tobac.add_D_equ(df_merged_anvil["area"])
    
    
    df_merged_anvil = use_tobac.add_bulk_stats(
        df_merged_anvil, segmentation_mask_anvil, dtiwp, stats=["sum"]
    )
    

    
    print('Bulk stats anvils calculated')

Bulk stats anvils calculated


## 1.4.  Save mask and final features

In [18]:
if filedoesnotexist: 
    # save segmentation mask results
    segmentation_mask_anvil_ds = xr.DataArray.from_iris(segmentation_mask_anvil)

# 2. Precip cell Detection and Segmentation on Precip image
- started with nbook from fabian warmworld 03
## 2.1. tobac detection on precip image

In [20]:
if filedoesnotexist: 
    # Perform feature detection:
    print('starting cell feature detection')
    features_cell = tobac.feature_detection_multithreshold(precip_iris, dxy, **feature_detection_params_cell)
    print('cell feature detection performed')

starting cell feature detection
cell feature detection performed


## 2.2. Precip cell segmentation
### filter features

In [21]:
if filedoesnotexist: 
    #filter for thres
    filtered_features_cell = features_cell[features_cell["threshold_value"] >= detection_threshold_cell]

### set segmentation üarameter

In [22]:
if filedoesnotexist: 
    # Dictionary containing keyword arguments for segmentation step:
    segmentation_params_cell={}
    segmentation_params_cell['method']='watershed'
    segmentation_params_cell['threshold']=segmentation_threshold_cell 
    segmentation_params_cell["target"] = "maximum"
    segmentation_params_cell["PBC_flag"]='hdim_2'


### perfrom segmentation

In [23]:
if filedoesnotexist: 
    # Perform segmentation:
    print('Starting precip segmentation.')
    segmentation_mask_cell, segmentation_features_cell =  tobac.segmentation.segmentation( #tobac.segmentation_2D(
        filtered_features_cell,precip_iris,dxy,**segmentation_params_cell)
    
    print('segmentation precip performed')

Starting precip segmentation.
segmentation precip performed


## 2.3. Add bulk stats as post processing

In [24]:
if filedoesnotexist: 
    import use_tobac
    
    # calc bulk plus area
    df_merged_cell = use_tobac.add_bulk_stats_cell(segmentation_features_cell, segmentation_mask_cell, precip_iris)
    
    
    df_merged_cell["d_eq"] = use_tobac.add_D_equ(df_merged_cell["area"])

    df_merged_cell = use_tobac.add_bulk_stats(
        df_merged_cell, segmentation_mask_cell, dtiwp, stats=["sum"]
    )
    
    df_merged_cell = use_tobac.add_bulk_stats(
        df_merged_cell, segmentation_mask_cell, iwp_iris, stats=["sum", "mean"]
    )

## 2.4. Save mask and final features

In [25]:
if filedoesnotexist: 
    # save segmentation mask results
    segmentation_mask_cell_ds = xr.DataArray.from_iris(segmentation_mask_cell)

# 3. save detect and seg parameters

# 4. add Anvil charactistics to precip cell features
- use both segmentation masks
- all precip cells that have an anvil above them anywhere, anvil ID covering the most precip cell pixels is selected

## 4.1 B (precip cell mask under anvil) 


In [27]:
if filedoesnotexist: 
    # create one dataset for masks 
    combi_mask = segmentation_mask_anvil_ds.to_dataset(name="segmentation_mask_anvil")
    combi_mask["segmentation_mask_cell"]=segmentation_mask_cell_ds
    combi_mask = combi_mask.squeeze()
    combi_mask
    
    # get cell ids (could also come from feature table)
    cell_ids = np.unique(combi_mask["segmentation_mask_cell"].values)
    cell_ids = cell_ids[cell_ids!=0]
    
    
    cell_to_anvil = {}
    diagnostic_anvils_per_cell = {}
    for cell_id in cell_ids:
        # get anvil values only where this cell is
        anvil_vals = combi_mask["segmentation_mask_anvil"].where(
            combi_mask["segmentation_mask_cell"] == cell_id).values.flatten()
        
        anvil_vals = anvil_vals[~np.isnan(anvil_vals) & (anvil_vals != 0)].astype(int)
    
        diagnostic_anvils_per_cell[int(cell_id)] = list(np.unique(anvil_vals).astype(int))
        
        if len(anvil_vals) > 0: # 1 to 1 match! 
            # pick anvil with most overlapping pixels
            unique, counts = np.unique(anvil_vals, return_counts=True)
            cell_to_anvil[int(cell_id)] = int(unique[np.argmax(counts)])
    
    ### add anvil look up table and update mask
    # add anvil id look up in combined cell features
    df_merged_cell["lookup_anvil_feature"] = df_merged_cell["feature"].map(cell_to_anvil).fillna(0).astype(int)
    
    
    # update filtered masks to reflect 1 to 1 map
    anvil_ids_final = set(cell_to_anvil.values())
    cell_ids_final = set(cell_to_anvil.keys())
    
    combi_mask["segmentation_mask_cell_filtered"] = combi_mask["segmentation_mask_cell"].where(
        combi_mask["segmentation_mask_cell"].isin(list(cell_ids_final)), other=0
    )
    combi_mask["segmentation_mask_anvil_filtered"] = combi_mask["segmentation_mask_anvil"].where(
        combi_mask["segmentation_mask_anvil"].isin(list(anvil_ids_final)), other=0
    )

## 4.2. join selected anvil features by anvil_idx
df_merged_anvil.columns

    Index(['frame', 'idx', 'hdim_1', 'hdim_2', 'num', 'threshold_value', 'feature', 'time', 'timestr', 'latitude', 'longitude', 'ncells', 'mean_iwp', 'total_iwp', 'max_iwp', 'mean_cell', 'total_cell', 'max_cell', 'min_tb', 'area', 'd_eq'], dtype='str')

In [28]:
if filedoesnotexist: 
    # select columns to join
    anvil_columns_to_join = ['idx', 'hdim_1', 'hdim_2','num', 'threshold_value', 'feature','latitude', 'longitude','ncells', 'mean_iwp',
           'total_iwp', 'max_iwp','min_tb', 'area', 'd_eq','mean_precip', 'total_precip', 'max_precip',"total_dtiwp"]
    df_merged_anvil_selection = df_merged_anvil[anvil_columns_to_join]
    
    #df_merged_anvil_selection
    df_merged_cell_combined = df_merged_cell.merge(df_merged_anvil_selection,
                                                       how = "inner" ,# left to keep all cell features # inner to only keep precip cells that have anvil
                                                       left_on = "lookup_anvil_feature",right_on = "feature",
                                                       suffixes=("_cell", "_anvil")) # change precip to cell => to distingish "variable precip" and precip CELL detection
    
    # drop lookup_anvil_feature column bc its the same as feature_anvil
    df_merged_cell_combined = df_merged_cell_combined.drop(columns=["lookup_anvil_feature"])

In [29]:
if filedoesnotexist: 
    import pandas as pd
    #solve issue with numeric columns / pytables warning 
    numeric_cols = [c for c in df_merged_cell_combined.columns if 'dtiwp' in c or 'mean_iwp_cell' in c]
    df_merged_cell_combined[numeric_cols] = df_merged_cell_combined[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 4.4 save 